<a href="https://colab.research.google.com/github/osergioribeirof/Python/blob/main/SR_GammaFlip_Interativo_Barchart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
### CÓDIGO ADAPTADO PARA FORMATO BARCHART ###

import pandas as pd
import plotly
pd.set_option('plotting.backend','plotly')
import plotly.graph_objs as go
import numpy as np
import scipy
from scipy.stats import norm
import calendar
from datetime import datetime, timedelta, date

In [9]:
# ========== CONFIGURAÇÕES ==========
# IMPORTANTE: Defina o preço spot atual do ativo
spotPrice = 26200.00  # AJUSTE AQUI O PREÇO ATUAL DO NQ

# Nome do arquivo CSV
csv_file = '/content/nqz25-volatility-greeks-exp-12_19_25-50-strikes-+_--10-29-2025.csv'

In [10]:
# ========== LEITURA E LIMPEZA DOS DADOS ==========
print("Lendo arquivo CSV...")
df = pd.read_csv(csv_file)

Lendo arquivo CSV...


In [11]:
# Remover última linha (metadados do Barchart)
df = df[df['Type'].notna()].copy()

# Limpar coluna Strike (remover vírgulas e converter para float)
df['Strike'] = df['Strike'].str.replace(',', '').astype(float)

# Limpar IV (remover % e converter para decimal)
df['IV'] = df['IV'].str.replace('%', '').astype(float) / 100

# Limpar IV Skew
df['IV Skew'] = df['IV Skew'].str.replace('%', '').str.replace('+', '').astype(float) / 100

print(f"✓ Dados carregados: {len(df)} linhas")
print(f"  - Calls: {len(df[df['Type']=='Call'])}")
print(f"  - Puts: {len(df[df['Type']=='Put'])}")

✓ Dados carregados: 160 linhas
  - Calls: 80
  - Puts: 80


In [12]:
# ========== ADICIONAR OPEN INTEREST ==========
# ATENÇÃO: O arquivo Barchart NÃO inclui Open Interest!
# Você tem 3 opções:

# OPÇÃO 1: Usar valor padrão (apenas para visualização de gamma)
df['OpenInterest'] = 100  # Valor padrão

# OPÇÃO 2: Adicionar manualmente de outra fonte
# df_oi = pd.read_csv('open_interest.csv')  # arquivo separado com OI
# df = df.merge(df_oi, on=['Strike', 'Type'], how='left')

# OPÇÃO 3: Estimar baseado em volume ou last (menos preciso)
# df['OpenInterest'] = df['Last'] / 10  # exemplo de estimativa

print(f"✓ Open Interest configurado (método: valor padrão)")

✓ Open Interest configurado (método: valor padrão)


In [13]:
# ========== SEPARAR CALLS E PUTS ==========
calls = df[df['Type'] == 'Call'].copy()
puts = df[df['Type'] == 'Put'].copy()

In [14]:
# ========== CALCULAR GEX ==========
# GEX = Gamma × Open Interest × 100 × Spot² × 0.01
# Calls têm GEX positivo, Puts negativo

calls['GEX'] = calls['Gamma'] * calls['OpenInterest'] * 100 * (spotPrice ** 2) * 0.01
puts['GEX'] = puts['Gamma'] * puts['OpenInterest'] * 100 * (spotPrice ** 2) * 0.01 * -1

print(f"✓ GEX calculado")


✓ GEX calculado


In [15]:
# ========== AGREGAR POR STRIKE ==========
# Combinar calls e puts
all_options = pd.concat([calls, puts])


In [16]:
# Agrupar por strike
gex_by_strike = all_options.groupby('Strike').agg({
    'GEX': 'sum',
    'Gamma': lambda x: x.abs().sum(),
    'Delta': 'sum',
    'OpenInterest': 'sum'
}).reset_index()

In [17]:
# Separar GEX positivo e negativo para colorir o gráfico
gex_by_strike['GEX_positive'] = gex_by_strike['GEX'].apply(lambda x: x if x > 0 else 0)
gex_by_strike['GEX_negative'] = gex_by_strike['GEX'].apply(lambda x: x if x < 0 else 0)

print(f"✓ Agregação concluída: {len(gex_by_strike)} strikes únicos")

✓ Agregação concluída: 80 strikes únicos


### IMPORTANTE

In [18]:
# ========== IDENTIFICAR NÍVEIS IMPORTANTES ==========
# Gamma Flip (onde GEX muda de sinal)
zero_crossings = []
for i in range(len(gex_by_strike)-1):
    if (gex_by_strike.iloc[i]['GEX'] > 0 and gex_by_strike.iloc[i+1]['GEX'] < 0) or \
       (gex_by_strike.iloc[i]['GEX'] < 0 and gex_by_strike.iloc[i+1]['GEX'] > 0):
        zero_crossings.append((gex_by_strike.iloc[i]['Strike'] + gex_by_strike.iloc[i+1]['Strike']) / 2)

# Maior GEX positivo (resistência)
max_gex_strike = gex_by_strike.loc[gex_by_strike['GEX'].idxmax(), 'Strike']
max_gex_value = gex_by_strike.loc[gex_by_strike['GEX'].idxmax(), 'GEX']

# Maior GEX negativo (suporte)
min_gex_strike = gex_by_strike.loc[gex_by_strike['GEX'].idxmin(), 'Strike']
min_gex_value = gex_by_strike.loc[gex_by_strike['GEX'].idxmin(), 'GEX']

print("\n=== NÍVEIS IMPORTANTES ===")
print(f"Spot Price: ${spotPrice:,.2f}")
print(f"Gamma Flip: {zero_crossings}")
print(f"Resistência (Max GEX+): ${max_gex_strike:,.2f} ({max_gex_value:,.0f})")
print(f"Suporte (Max GEX-): ${min_gex_strike:,.2f} ({min_gex_value:,.0f})")


=== NÍVEIS IMPORTANTES ===
Spot Price: $26,200.00
Gamma Flip: [np.float64(24350.0), np.float64(28125.0)]
Resistência (Max GEX+): $25,900.00 (5,619,893)
Suporte (Max GEX-): $23,300.00 (-3,767,968)


In [19]:
# ========== GRÁFICO 1: GEX POR STRIKE ==========
fig1 = go.Figure()

# Barras positivas (verde)
fig1.add_trace(go.Bar(
    x=gex_by_strike['Strike'],
    y=gex_by_strike['GEX_positive'],
    name='Call GEX (Resistência)',
    marker_color='green',
    opacity=0.7
))

# Barras negativas (vermelho)
fig1.add_trace(go.Bar(
    x=gex_by_strike['Strike'],
    y=gex_by_strike['GEX_negative'],
    name='Put GEX (Suporte)',
    marker_color='red',
    opacity=0.7
))

# Linha do spot price
fig1.add_vline(x=spotPrice, line_dash="dash", line_color="blue",
               annotation_text=f"Spot: ${spotPrice:,.0f}")

# Linhas de gamma flip
for flip in zero_crossings:
    fig1.add_vline(x=flip, line_dash="dot", line_color="orange",
                   annotation_text=f"Flip: ${flip:,.0f}")

fig1.update_layout(
    title='Gamma Exposure (GEX) por Strike - NQ Futuro',
    xaxis_title='Strike Price',
    yaxis_title='GEX',
    hovermode='x unified',
    template='plotly_dark',
    barmode='overlay'
)

fig1.show()




In [20]:
# ========== GRÁFICO 2: DELTA POR STRIKE ==========
calls_grouped = calls.groupby('Strike')['Delta'].sum().reset_index()
puts_grouped = puts.groupby('Strike')['Delta'].sum().reset_index()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=calls_grouped['Strike'],
    y=calls_grouped['Delta'],
    name='Call Delta',
    line=dict(color='green', width=2)
))

fig2.add_trace(go.Scatter(
    x=puts_grouped['Strike'],
    y=puts_grouped['Delta'],
    name='Put Delta',
    line=dict(color='red', width=2)
))

fig2.add_vline(x=spotPrice, line_dash="dash", line_color="blue",
               annotation_text=f"Spot: ${spotPrice:,.0f}")

fig2.update_layout(
    title='Delta Exposure por Strike - NQ Futuro',
    xaxis_title='Strike Price',
    yaxis_title='Delta Total',
    hovermode='x unified',
    template='plotly_dark'
)

fig2.show()

In [21]:
# ========== GRÁFICO 3: IMPLIED VOLATILITY SKEW ==========
fig3 = go.Figure()

fig3.add_trace(go.Scatter(
    x=calls['Strike'],
    y=calls['IV'] * 100,
    mode='markers+lines',
    name='Call IV',
    marker=dict(color='green', size=6)
))

fig3.add_trace(go.Scatter(
    x=puts['Strike'],
    y=puts['IV'] * 100,
    mode='markers+lines',
    name='Put IV',
    marker=dict(color='red', size=6)
))

fig3.add_vline(x=spotPrice, line_dash="dash", line_color="blue",
               annotation_text=f"Spot: ${spotPrice:,.0f}")

fig3.update_layout(
    title='Volatilidade Implícita (IV) por Strike - NQ Futuro',
    xaxis_title='Strike Price',
    yaxis_title='IV (%)',
    hovermode='x unified',
    template='plotly_dark'
)

fig3.show()



In [22]:
# ========== GRÁFICO 4: GAMMA PROFILE ==========
gamma_calls = calls.groupby('Strike')['Gamma'].sum().reset_index()
gamma_puts = puts.groupby('Strike')['Gamma'].sum().reset_index()

fig4 = go.Figure()

fig4.add_trace(go.Bar(
    x=gamma_calls['Strike'],
    y=gamma_calls['Gamma'],
    name='Call Gamma',
    marker_color='green',
    opacity=0.6
))

fig4.add_trace(go.Bar(
    x=gamma_puts['Strike'],
    y=gamma_puts['Gamma'],
    name='Put Gamma',
    marker_color='red',
    opacity=0.6
))

fig4.add_vline(x=spotPrice, line_dash="dash", line_color="blue",
               annotation_text=f"Spot: ${spotPrice:,.0f}")

fig4.update_layout(
    title='Perfil de Gamma por Strike - NQ Futuro',
    xaxis_title='Strike Price',
    yaxis_title='Gamma Total',
    hovermode='x unified',
    template='plotly_dark',
    barmode='group'
)

fig4.show()

In [23]:
# ========== ESTATÍSTICAS FINAIS ==========
print("\n=== ESTATÍSTICAS FINAIS ===")
print(f"Total GEX: {gex_by_strike['GEX'].sum():,.0f}")
print(f"GEX Positivo (Call): {gex_by_strike['GEX_positive'].sum():,.0f}")
print(f"GEX Negativo (Put): {gex_by_strike['GEX_negative'].sum():,.0f}")
print(f"Ratio Call/Put GEX: {abs(gex_by_strike['GEX_positive'].sum() / gex_by_strike['GEX_negative'].sum()):.2f}")

print("\n✓ Análise concluída!")


=== ESTATÍSTICAS FINAIS ===
Total GEX: -42,533,531
GEX Positivo (Call): 58,695,932
GEX Negativo (Put): -101,229,463
Ratio Call/Put GEX: 0.58

✓ Análise concluída!
